In [ ]:
! pip install -q transformers sentence-transformers accelerate pypdf chromadb langchain-text-splitters

In [ ]:
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
import pypdf
from pypdf import PdfReader
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
reader = PdfReader("doc4_travel_expense_policy.pdf")
pdf_text = ""
for rec in reader.pages:
   pdf_text += rec.extract_text()

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(

    chunk_size=300,

    chunk_overlap=50,

    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]

)

chunks = text_splitter.split_text(pdf_text)

print("Total Chunks :", len(chunks))

Total Chunks : 5


In [ ]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = embedding_model.encode(chunks)

In [ ]:
chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)
collection = chroma_client.create_collection(name="my_collections")

In [ ]:
ids = []

for i in range(len(chunks)):
    ids.append(str(i))

In [ ]:
collection.add(

    ids=ids,

    documents=chunks,

    embeddings=embeddings.tolist()

)
print(collection.count())

In [ ]:

tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct"
)

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct"
)

pipe = pipeline(

    "text-generation",

    model=model,

    tokenizer=tokenizer
)

In [ ]:
def rag_tool(question):
    question_embedding = embedding_model.encode([question])
    results = collection.query(
        query_embeddings=question_embedding.tolist(),
        n_results = 3
    )
    retrieved_chunks = results['documents'][0]
    context ="\n\n".join(retrieved_chunks)
    return context

def calculator_tool(question):

    expression = question.lower()

    expression = expression.replace("what is", "")
    expression = expression.replace("calculate", "")
    expression = expression.replace("=", "")

    expression = expression.replace("×", "*")
    expression = expression.replace("x", "*")
    expression = expression.replace("÷", "/")

    expression = expression.strip()

    answer = eval(expression)

    return answer


def greeting_tool():

    return """
Hello!

I can help you with:

1. Answering questions from your PDF documents.
2. Solving mathematical calculations.

How can I help you today?
"""

def choose_tool(question):

    question = question.lower().strip()

    # Greeting Tool
    if question in ["hi", "hello", "hey", "good morning", "good evening"]:
        return "greeting"

    # Calculator Tool
    elif any(operator in question for operator in ["+", "-", "*", "/", "×", "÷"]):
        return "calculator"

    # Otherwise use RAG
    else:
        return "rag"

while True:

    question = input(
        "Enter your question (enter 'exit' if you don't want to continue): "
    )

    if question.lower() == "exit":
        break

    tool = choose_tool(question)

    if tool == "greeting":

        answer = greeting_tool()

    elif tool == "calculator":

        answer = calculator_tool(question)

    elif tool == "rag":

        context = rag_tool(question)

        prompt = f"""<|user|>
Use ONLY the context below to answer the question.

Context:
{context}

Question:
{question}

If the answer is not present, reply exactly:
I couldn't find that information.
<|end|>

<|assistant|>
"""

        result = pipe(
            prompt,
            max_new_tokens=120,
            return_full_text=False
        )

        answer = result[0]["generated_text"].strip()

    print("\nAnswer:")
    print(answer)
    print("-" * 60)